In [1]:
import re
import emoji
import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# Scarica le stopwords italiane
nltk.download('stopwords', quiet=True)
stop_words_ita = set(stopwords.words('italian'))

recensioni = [
    # Prodotto fisico / Tech
    "Spedizione velocissima! 📦🚚 Pacco arrivato intatto ✨... Le cuffie suonano bene 🎧, ma il cavo è davvero TROPPO corto!! 😡 Voto: 4/5 ★★★★☆",
    "Pessimo! ❌ Dò 1/5 ⭐️. Dopo appena 2 giorni (48h!) ha smesso di funzionare 💥. Ho avviato il reso... #Deluso 💸👎",
    "Rapporto qualità/prezzo TOP!! 🔥⚡️ Materiali 100% resistenti e design curato 👌. Consigliatissimo @tutti 💯💯",
    "Servizio ok 🤷‍♂️, ma la qualità *non* rispecchia le foto... 📉 Piuttosto deluso ~_~ (prezzo alto vs qualità mediocre) 💰➡️🗑️",

    # Ristorante / Food
    "Cibo ECCELLENTE!! 🍕🍷 Ingredienti 100% freschi 🌿. Personale super cortese ❤️, anche se ~40 min di attesa... ⏳️",
    "Pizza ottima 😋 e ben lievitata 👏! Però locale rumoroso (troppo BASS!!) 📢 e tavoli appiccicati... 🛋️=🛋️ Voto 3.5/5 ★★★☆☆",

    # Servizi / Hotel
    "Soggiorno PER-FET-TO! 🏨✨ Camera pulitissima 🧹& colazione SUPER abbondante 🥐☕️. Torneremo sicuramente!! ✈️🇮🇹",
    "Assistenza clienti pessima 📧❌... Lenta & poco risolutiva! Ho chiamato x3 volte! ☎️💥 #PessimoServizio"
]

def pulisci_e_tokenizza(frase):
    # 1. Rimuovo le emoji
    testo_no_emoji = emoji.replace_emoji(frase, replace="")

    # 2. Lowercasing
    testo_lower = testo_no_emoji.lower()

    # 3. Rimozione della punteggiatura e caratteri speciali
    testo_no_punteggiatura = re.sub(r'[^\w\s]', '', testo_lower)

    # 4. Tokenizzazione (split sulle parole)
    token = testo_no_punteggiatura.split()

    # 5. Rimozione delle stopwords e parole composte solo da numeri
    token_puliti = [parola for parola in token if parola not in stop_words_ita and not parola.isdigit()]

    return token_puliti

# Applicazione del processo a tutte le recensioni
token_per_recensione = [pulisci_e_tokenizza(r) for r in recensioni]

# Estrazione di tutti i token in un'unica lista flat
tutti_i_token = [token for sottolista in token_per_recensione for token in sottolista]

# Calcolo token totali e unici
token_totali = len(tutti_i_token)
token_unici = set(tutti_i_token)


print(f"Token totali (dopo pulizia): {token_totali}")
print(f"Token unici (Vocabolario): {len(token_unici)}\n")

print("Esempio di tokenizzazione sulla prima recensione:")
print(f"Originale: {recensioni[0]}")
print(f"Token filtrati: {token_per_recensione[0]}")



Token totali (dopo pulizia): 85
Token unici (Vocabolario): 80

Esempio di tokenizzazione sulla prima recensione:
Originale: Spedizione velocissima! 📦🚚 Pacco arrivato intatto ✨... Le cuffie suonano bene 🎧, ma il cavo è davvero TROPPO corto!! 😡 Voto: 4/5 ★★★★☆
Token filtrati: ['spedizione', 'velocissima', 'pacco', 'arrivato', 'intatto', 'cuffie', 'suonano', 'bene', 'cavo', 'davvero', 'troppo', 'corto', 'voto']


## Stemming e lammatization sulla stssa frase per il confronto

In [2]:
# 1. Inizializza lo stemmer specificando la lingua italiana
stemmer_italiano = SnowballStemmer("italian")

# token_filtrati

token_filtrati = [
    'spedizione', 'velocissima', 'pacco', 'arrivato', 'intatto', 
    'cuffie', 'suonano', 'bene', 'cavo', 'davvero', 'troppo', 
    'corto', 'voto'
]

# 3. Applicare lo stemming a ogni singola parola della lista
token_stemmati = [stemmer_italiano.stem(token) for token in token_filtrati]

print("Token originali:", token_filtrati)
print("Token stemmati :", token_stemmati)

Token originali: ['spedizione', 'velocissima', 'pacco', 'arrivato', 'intatto', 'cuffie', 'suonano', 'bene', 'cavo', 'davvero', 'troppo', 'corto', 'voto']
Token stemmati : ['spedizion', 'velocissim', 'pacc', 'arriv', 'intatt', 'cuff', 'suon', 'ben', 'cav', 'davver', 'tropp', 'cort', 'vot']


# Lemmatization 

In [3]:
import spacy
nlp = spacy.load("it_core_news_sm")

# frase originale prima della pulizia
frase_originale = ["Spedizione velocissima! 📦🚚 Pacco arrivato intatto ✨... Le cuffie suonano bene 🎧, ma il cavo è davvero TROPPO corto!! 😡 Voto: 4/5 ★★★★☆"]

# 3. Passa la frase a spaCy per l'elaborazione
doc = nlp(frase_originale)

# 4. Filtra i token (rimuovendo punteggiatura, emoji e stopwords) e prendi il lemma

token_lemmatizzati = [
    token.lemma_.lower()                       # Prende il lemma in minuscolo
    for token in doc 
    if not token.is_punct                      # Rimuove la punteggiatura (!, :, ...)
    and not token.is_stop                      # Rimuove le stopwords (le, ma, il, è)
    and token.text.strip()                     # Rimuove spazi vuoti
    and not token.like_num                     # Rimuove i numeri (4, 5)
    and token.pos_ not in ["PUNCT", "SYM"]     # Rimuove simboli ed emoji (📦, 🎧, ★)
]

print("Token lemmatizzati:", token_lemmatizzati)

OSError: [E050] Can't find model 'it_core_news_sm'. It doesn't seem to be a Python package or a valid path to a data directory.